# LandmarkDiff Advanced Usage

This notebook covers advanced functionality beyond the quickstart:

1. Custom procedure creation with DeformationHandle
2. Data-driven displacements via DisplacementModel
3. Clinical flags (vitiligo, Bell's palsy, keloid, Ehlers-Danlos)
4. Multi-procedure combinations
5. Intensity sweeps and visualization
6. Post-processing customization
7. Saving and comparing results

**Prerequisites:** `pip install -e .` from the repo root. Some cells require a GPU for ControlNet mode.

In [ ]:
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from landmarkdiff.conditioning import auto_canny, render_wireframe

# core imports
from landmarkdiff.landmarks import (
    FaceLandmarks,
    extract_landmarks,
    render_landmark_image,
    visualize_landmarks,
)
from landmarkdiff.manipulation import (
    PROCEDURE_LANDMARKS,
    PROCEDURE_RADIUS,
    DeformationHandle,
    apply_procedure_preset,
    gaussian_rbf_deform,
)

In [ ]:
# load a sample face image
sample_url = (
    "https://raw.githubusercontent.com/NVlabs/ffhq-dataset/master/thumbnails128x128/00000.png"
)
sample_path = Path("sample_face.png")

if not sample_path.exists():
    urllib.request.urlretrieve(sample_url, sample_path)

img = np.array(Image.open(sample_path).convert("RGB").resize((512, 512)))
# convert RGB -> BGR for OpenCV/MediaPipe
img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

face = extract_landmarks(img_bgr)
assert face is not None, "No face detected in sample image"

print(f"Detected {face.landmarks.shape[0]} landmarks")
print(f"Image size: {face.image_width}x{face.image_height}")
print(f"Confidence: {face.confidence:.2f}")

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title("input")
plt.axis("off")
plt.show()

## 1. Custom Procedure Creation

You can define any deformation by specifying:
- **landmark_index**: which of the 478 MediaPipe landmarks acts as the control point
- **displacement**: a 2D (or 3D) pixel displacement vector `[dx, dy]`
- **influence_radius**: the Gaussian RBF sigma in pixels (larger = wider area affected)

The `gaussian_rbf_deform` function applies `delta * exp(-dist^2 / 2r^2)` to all landmarks.

In [ ]:
# define a custom lip augmentation procedure
# each handle is a control point with a displacement vector and influence radius
lip_handles = [
    # upper lip center - push upward
    DeformationHandle(landmark_index=13, displacement=np.array([0.0, -3.0]), influence_radius=15.0),
    DeformationHandle(landmark_index=14, displacement=np.array([0.0, -3.0]), influence_radius=15.0),
    # upper lip lateral
    DeformationHandle(landmark_index=82, displacement=np.array([0.0, -2.0]), influence_radius=12.0),
    DeformationHandle(
        landmark_index=312, displacement=np.array([0.0, -2.0]), influence_radius=12.0
    ),
    # lower lip - push downward
    DeformationHandle(landmark_index=17, displacement=np.array([0.0, 2.0]), influence_radius=15.0),
    # lip corners - slight lift
    DeformationHandle(landmark_index=61, displacement=np.array([0.0, -2.0]), influence_radius=10.0),
    DeformationHandle(
        landmark_index=291, displacement=np.array([0.0, -2.0]), influence_radius=10.0
    ),
]

# pixel_coords is a @property, not a method
pixel_lm = face.pixel_coords
print(f"pixel_coords shape: {pixel_lm.shape}")  # (478, 2)

In [ ]:
# apply handles at multiple intensity levels
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# show original wireframe
original_wire = render_wireframe(face, width=512, height=512)
axes[0].imshow(original_wire, cmap="gray")
axes[0].set_title("original")
axes[0].axis("off")

for col, intensity in enumerate([30, 60, 90], start=1):
    scale = intensity / 100.0
    # start from the full normalized landmark array
    deformed_pts = face.landmarks.copy()

    # convert to pixel space for deformation
    deformed_pts[:, 0] *= face.image_width
    deformed_pts[:, 1] *= face.image_height

    for handle in lip_handles:
        scaled_handle = DeformationHandle(
            landmark_index=handle.landmark_index,
            displacement=handle.displacement * scale,
            influence_radius=handle.influence_radius,
        )
        deformed_pts = gaussian_rbf_deform(deformed_pts, scaled_handle)

    # convert back to normalized
    deformed_pts[:, 0] /= face.image_width
    deformed_pts[:, 1] /= face.image_height

    deformed_face = FaceLandmarks(
        landmarks=deformed_pts,
        image_width=face.image_width,
        image_height=face.image_height,
        confidence=face.confidence,
    )

    wire = render_wireframe(deformed_face, width=512, height=512)
    axes[col].imshow(wire, cmap="gray")
    axes[col].set_title(f"lip augmentation @ {intensity}%")
    axes[col].axis("off")

plt.tight_layout()
plt.show()

### Inspecting built-in procedure landmarks

Each built-in procedure has a set of targeted landmark indices and a default influence radius.
You can use these as a starting point for custom procedures.

In [ ]:
for proc, indices in PROCEDURE_LANDMARKS.items():
    radius = PROCEDURE_RADIUS[proc]
    top5 = indices[:5]
    print(f"{proc:20s}  landmarks: {len(indices):3d}  radius: {radius:.0f}px  indices[:5]: {top5}")

## 2. Data-Driven Displacements with DisplacementModel

Instead of hand-tuned RBF vectors, you can fit a `DisplacementModel` from real surgery pairs.
The model learns per-landmark mean and variance of displacement vectors grouped by procedure.

When passed to `apply_procedure_preset` via the `displacement_model_path` parameter,
it replaces the built-in handles with statistically learned displacements.

In [ ]:
from landmarkdiff.displacement_model import DisplacementModel

# create a synthetic displacement model for demonstration
# in practice, you would fit from real surgery pairs using extract_from_directory()
rng = np.random.default_rng(42)

synthetic_data = []
for _ in range(20):
    # simulate a rhinoplasty displacement pattern:
    # nose landmarks move, everything else stays roughly still
    disp = rng.normal(0, 0.001, size=(478, 2)).astype(np.float32)
    nose_indices = PROCEDURE_LANDMARKS["rhinoplasty"]
    for idx in nose_indices:
        disp[idx] = rng.normal(0, 0.005, size=2)  # larger displacement in nose region
    synthetic_data.append({"displacements": disp, "procedure": "rhinoplasty"})

model = DisplacementModel()
model.fit(synthetic_data)

print(f"Fitted procedures: {model.procedures}")
print(f"Samples per procedure: {model.n_samples}")
print(f"Model fitted: {model.fitted}")

In [ ]:
# generate a displacement field and inspect it
field = model.get_displacement_field(
    procedure="rhinoplasty",
    intensity=1.0,  # 1.0 = mean observed displacement
    noise_scale=0.0,  # 0 = deterministic (no variation)
)
print(f"Displacement field shape: {field.shape}")  # (478, 2)
print(f"Mean magnitude: {np.mean(np.linalg.norm(field, axis=1)):.5f}")

# show top-10 most displaced landmarks
summary = model.get_summary("rhinoplasty")
top_lm = summary["procedures"]["rhinoplasty"]["top_landmarks"]
print("\nTop 10 landmarks by mean displacement magnitude:")
for entry in top_lm:
    print(f"  landmark {entry['index']:3d}: {entry['magnitude']:.5f}")

In [ ]:
# save and reload the model
model_path = Path("demo_displacement_model.npz")
model.save(model_path)
print(f"Saved model to {model_path}")

reloaded = DisplacementModel.load(model_path)
print(f"Reloaded procedures: {reloaded.procedures}")
print(f"Reloaded samples: {reloaded.n_samples}")

In [ ]:
# use the saved model with apply_procedure_preset
# this replaces hand-tuned handles with data-driven displacements
deformed_dd = apply_procedure_preset(
    face,
    procedure="rhinoplasty",
    intensity=70,  # 0-100 scale
    displacement_model_path=str(model_path),
    noise_scale=0.3,  # add some variation
)

# compare hand-tuned vs data-driven
deformed_ht = apply_procedure_preset(face, "rhinoplasty", intensity=70)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(render_wireframe(face, width=512, height=512), cmap="gray")
axes[0].set_title("original")
axes[0].axis("off")

axes[1].imshow(render_wireframe(deformed_ht, width=512, height=512), cmap="gray")
axes[1].set_title("hand-tuned RBF")
axes[1].axis("off")

axes[2].imshow(render_wireframe(deformed_dd, width=512, height=512), cmap="gray")
axes[2].set_title("data-driven")
axes[2].axis("off")

plt.suptitle("rhinoplasty @ 70%: hand-tuned vs data-driven", fontsize=13)
plt.tight_layout()
plt.show()

# cleanup
model_path.unlink(missing_ok=True)

## 3. Clinical Flags

LandmarkDiff supports four clinical conditions that modify deformation behavior:

| Flag | Effect |
|------|--------|
| `vitiligo` | Detects and preserves depigmented patches during compositing |
| `bells_palsy` | Disables deformation on the paralyzed side |
| `keloid_prone` | Softens mask boundaries near incision-prone regions |
| `ehlers_danlos` | Widens RBF influence radius by 1.5x for hypermobile tissue |

In [ ]:
from landmarkdiff.clinical import ClinicalFlags

# Ehlers-Danlos: tissue is hypermobile, so deformations spread wider
flags_eds = ClinicalFlags(ehlers_danlos=True)

deformed_normal = apply_procedure_preset(face, "rhinoplasty", intensity=60)
deformed_eds = apply_procedure_preset(face, "rhinoplasty", intensity=60, clinical_flags=flags_eds)

# the Ehlers-Danlos flag multiplies influence radius by 1.5x
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(render_wireframe(face, width=512, height=512), cmap="gray")
axes[0].set_title("original")
axes[0].axis("off")

axes[1].imshow(render_wireframe(deformed_normal, width=512, height=512), cmap="gray")
axes[1].set_title("standard rhinoplasty")
axes[1].axis("off")

axes[2].imshow(render_wireframe(deformed_eds, width=512, height=512), cmap="gray")
axes[2].set_title("with Ehlers-Danlos")
axes[2].axis("off")

plt.suptitle("effect of Ehlers-Danlos flag on deformation spread", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Bell's palsy: only deform the healthy side
flags_bp = ClinicalFlags(bells_palsy=True, bells_palsy_side="left")

deformed_bp = apply_procedure_preset(face, "rhytidectomy", intensity=60, clinical_flags=flags_bp)
deformed_standard = apply_procedure_preset(face, "rhytidectomy", intensity=60)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(render_wireframe(face, width=512, height=512), cmap="gray")
axes[0].set_title("original")
axes[0].axis("off")

axes[1].imshow(render_wireframe(deformed_standard, width=512, height=512), cmap="gray")
axes[1].set_title("bilateral rhytidectomy")
axes[1].axis("off")

axes[2].imshow(render_wireframe(deformed_bp, width=512, height=512), cmap="gray")
axes[2].set_title("Bell's palsy (left affected)")
axes[2].axis("off")

plt.suptitle("Bell's palsy: paralyzed side excluded from deformation", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# keloid-prone patient: detects and softens mask transitions in prone regions
from landmarkdiff.clinical import get_keloid_exclusion_mask

flags_keloid = ClinicalFlags(keloid_prone=True, keloid_regions=["nose", "jawline"])

# visualize the keloid exclusion mask
keloid_mask = get_keloid_exclusion_mask(
    face, flags_keloid.keloid_regions, width=512, height=512, margin_px=10
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img)
axes[0].set_title("input")
axes[0].axis("off")

axes[1].imshow(keloid_mask, cmap="hot")
axes[1].set_title("keloid-prone regions (nose + jawline)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print(f"Flags active: {flags_keloid.has_any()}")

In [ ]:
# vitiligo detection (best demonstrated on images with actual depigmented patches)
from landmarkdiff.clinical import detect_vitiligo_patches

vitiligo_mask = detect_vitiligo_patches(img_bgr, face, l_threshold=85.0, min_patch_area=200)
print(f"Vitiligo pixels detected: {np.count_nonzero(vitiligo_mask)}")
print("(Most sample faces won't have vitiligo patches -- the detection is conservative)")

# if you had patches, adjust_mask_for_vitiligo reduces blending over them:
# adjusted = adjust_mask_for_vitiligo(surgical_mask, vitiligo_mask, preservation_factor=0.3)

## 4. Multi-Procedure Combinations

You can chain multiple procedures by applying them sequentially.
Each call to `apply_procedure_preset` returns a new `FaceLandmarks`, so you
can feed the output of one procedure into the next.

In [ ]:
# combine rhinoplasty + brow_lift
step1 = apply_procedure_preset(face, "rhinoplasty", intensity=50)
step2 = apply_procedure_preset(step1, "brow_lift", intensity=40)

# or a three-procedure combination
combo = apply_procedure_preset(face, "rhinoplasty", intensity=40)
combo = apply_procedure_preset(combo, "blepharoplasty", intensity=50)
combo = apply_procedure_preset(combo, "mentoplasty", intensity=30)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

pairs = [
    (face, "original"),
    (step1, "rhinoplasty only"),
    (step2, "+ brow_lift"),
    (combo, "rhino + bleph + mento"),
]
for ax, (f, title) in zip(axes, pairs, strict=False):
    ax.imshow(render_wireframe(f, width=512, height=512), cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("multi-procedure combinations", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Intensity Sweeps and Visualization

The intensity parameter runs from 0 (no change) to 100 (maximum deformation).
Internally, `apply_procedure_preset` scales the displacement vectors by `intensity / 100`.

In [ ]:
# sweep across all 6 procedures at multiple intensities
procedures = list(PROCEDURE_LANDMARKS.keys())
intensities = [20, 40, 60, 80, 100]

fig, axes = plt.subplots(len(procedures), len(intensities), figsize=(20, 24))

for row, proc in enumerate(procedures):
    for col, intensity in enumerate(intensities):
        deformed = apply_procedure_preset(face, proc, intensity=intensity)
        wire = render_wireframe(deformed, width=512, height=512)
        axes[row, col].imshow(wire, cmap="gray")
        if row == 0:
            axes[row, col].set_title(f"{intensity}%", fontsize=12)
        if col == 0:
            axes[row, col].set_ylabel(proc, fontsize=12, rotation=0, labelpad=80)
        axes[row, col].axis("off")

plt.suptitle("intensity sweep across all procedures", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# visualize which landmarks each procedure affects
# overlay procedure-specific landmarks on the original face

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

pixel_pts = face.pixel_coords  # @property, returns (478, 2)

for i, (proc, indices) in enumerate(PROCEDURE_LANDMARKS.items()):
    canvas = np.zeros((512, 512, 3), dtype=np.uint8)

    # draw all landmarks as dim dots
    for x, y in pixel_pts:
        cv2.circle(canvas, (int(x), int(y)), 1, (60, 60, 60), -1)

    # highlight procedure-specific landmarks in green
    for idx in indices:
        x, y = pixel_pts[idx]
        cv2.circle(canvas, (int(x), int(y)), 3, (0, 255, 0), -1)

    axes[i].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"{proc} ({len(indices)} landmarks)")
    axes[i].axis("off")

plt.suptitle("targeted landmarks per procedure (green)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Post-Processing Customization

After deformation, you can generate different conditioning signals:
- **render_wireframe**: static anatomical adjacency wireframe (grayscale)
- **render_landmark_image**: full 2556-edge tessellation mesh (BGR, used by CrucibleAI ControlNet)
- **auto_canny**: adaptive Canny edge detection on the wireframe (Fitzpatrick I-VI safe)

You can also combine these with the original image for visualization.

In [ ]:
deformed = apply_procedure_preset(face, "rhinoplasty", intensity=60)

# different conditioning outputs
wireframe = render_wireframe(deformed, width=512, height=512)
mesh_img = render_landmark_image(deformed, width=512, height=512)
canny_edges = auto_canny(wireframe)

# landmark overlay on the original image
overlay = visualize_landmarks(img_bgr, deformed, radius=2, draw_regions=True)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(wireframe, cmap="gray")
axes[0].set_title("static wireframe")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(mesh_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("tessellation mesh")
axes[1].axis("off")

axes[2].imshow(canny_edges, cmap="gray")
axes[2].set_title("auto-canny edges")
axes[2].axis("off")

axes[3].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[3].set_title("colored landmark overlay")
axes[3].axis("off")

plt.suptitle("conditioning signal variants", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# full conditioning pipeline
from landmarkdiff.conditioning import generate_conditioning

landmark_img, canny, wire = generate_conditioning(deformed, width=512, height=512)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(landmark_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("landmarks (channel 1)")
axes[0].axis("off")

axes[1].imshow(canny, cmap="gray")
axes[1].set_title("canny edges (channel 2)")
axes[1].axis("off")

axes[2].imshow(wire, cmap="gray")
axes[2].set_title("wireframe (channel 3)")
axes[2].axis("off")

plt.suptitle("generate_conditioning() output channels", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Saving and Comparing Results

Save deformed wireframes and conditioning signals to disk for batch processing
or downstream use with ControlNet.

In [ ]:
output_dir = Path("advanced_output")
output_dir.mkdir(exist_ok=True)

# save conditioning for each procedure at a fixed intensity
for proc in PROCEDURE_LANDMARKS:
    deformed = apply_procedure_preset(face, proc, intensity=60)
    wire = render_wireframe(deformed, width=512, height=512)
    cv2.imwrite(str(output_dir / f"{proc}_wireframe.png"), wire)

print(f"Saved {len(PROCEDURE_LANDMARKS)} wireframes to {output_dir}/")
print("Files:", sorted(p.name for p in output_dir.glob("*.png")))

In [ ]:
# quantitative comparison: compute per-landmark displacement magnitude
# between original and deformed landmarks
deformed = apply_procedure_preset(face, "rhinoplasty", intensity=60)

diff = deformed.pixel_coords - face.pixel_coords  # both are @property
magnitudes = np.linalg.norm(diff, axis=1)

print(f"Mean displacement: {np.mean(magnitudes):.2f} px")
print(f"Max displacement:  {np.max(magnitudes):.2f} px")
print(f"Landmarks moved (>0.5px): {np.sum(magnitudes > 0.5)}")

# histogram of displacement magnitudes
plt.figure(figsize=(8, 4))
plt.hist(magnitudes, bins=50, color="steelblue", edgecolor="white")
plt.xlabel("displacement magnitude (px)")
plt.ylabel("landmark count")
plt.title("rhinoplasty @ 60% -- per-landmark displacement distribution")
plt.tight_layout()
plt.show()

In [ ]:
# cleanup demo outputs
import shutil

if output_dir.exists():
    shutil.rmtree(output_dir)
    print(f"Cleaned up {output_dir}/")

## Next steps

- See the [procedure guides](../docs/procedures/) for detailed anatomy and parameter recommendations per procedure
- Use `LandmarkDiffPipeline` for end-to-end inference with TPS or ControlNet backends
- Fit a `DisplacementModel` from real surgery image pairs for your own procedure types
- Check the [API reference](../docs/api/) for complete function signatures